# Praca domowa nr 2

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("allegro-api-transactions.csv")

In [ ]:
df.info()

Na szcz??cie braku nulli :)

In [ ]:
df.head()

In [ ]:
df[df['it_location'].str.startswith('warszawa')].it_location.value_counts()

In [ ]:
df[df['it_location'].str.startswith('Warszawa')].it_location.value_counts()

Wszystkie nazwy miast mo?na sprowadzi? do ma?ych liter - dzi?ki temu unikniemy kilku powt?rze?. Jednak, ta operacja i tak i tak nie wyeliminuje wszystkich powt?rze? spowodowanych b??dnym wpisaniem lokalizacji (w przypadku Warszawy mamy ponad 200 mo?liwo?ci). Nadal zostan? rekordy z wpisanymi kilkoma miastami (np. *warszawa, ??d?*) oraz konkretnymi adresami (np. *Warszawa ul.?opusza?ska 38 bud C*) lub innymi dodatkami, jak *??d?, tylko wysy?ka z magazynu*  czy *??d?, wysy?ka na ca?? Polsk?*.



In [ ]:
df[df['it_location'].str.startswith('??d?')].it_location.value_counts()

Mo?na by by?o przypisa? pierwsze s?owo (lub s?owa w przypadku miast o nazwach dwucz?onowych, np. Aleksandr?w ??dzki i Aleksandr?w Kujawski) jako adres, jednak najpierw potrzebna by by?a lista wszystkich poprawnych nazw miast, aby np. Aleksandr?w ??dzki i Aleksandr?w Kujawski nie sta?y si? jednym miastem. Wtedy mo?na by by?o r?wnie? wyeliminowa? liter?wki, takie jak *??d?* i *Lodz*.

In [ ]:
df[df['it_location'].str.startswith('Lodz')].it_location.value_counts()

In [ ]:
df[df['it_location'].str.startswith('Aleksandr?w')].it_location.value_counts()

In [ ]:
df['it_location'] = df['it_location'].str.lower()

## Target encoding dla zmiennej it_location

Link do animacji pokazuj?cej zasad? dzia?ania target encodingu: https://brendanhasz.github.io/2019/03/04/target-encoding#target-encoding

In [ ]:
import category_encoders as ce

dfX = df.copy()
encoder = ce.TargetEncoder(cols = 'it_location')

dfX['it_location'] = dfX['it_location'].str.lower()

dfX['it_location'] = encoder.fit_transform(dfX['it_location'], dfX['price'])
dfX.head()

Zalety: 
* kodowanie nie zwi?ksza wymiaru ramki danych - w tym przypadku przy zastosowaniu klasycznego One Hot Encoding dostaliby?my ramk? danych o ponad 10000 kolumnach

Wady:
* zale?ny od rozk?adu targetu dla danej kategorii - mo?e prowadzi? do przeuczenia, bo w zbiorze testowym/walidacyjnym dana kategoria mo?e mie? inny rozk?ad
* w kategoriach, w kt?rych mamy ma?o obserwacji, ?rednia nie b?dzie dobrym parametrem - du?e ryzyko przeuczenia
* mo?e by? niejednoznaczne - w szczeg?lno?ci w przypadku du?ej liczby r??nych klas - tracimy informacje o kategoriach, kt?re s? jakby grupowane wed?ug ?redniej: tak jak w tym przyk?adzie:

In [ ]:
print(f"Liczba unikalnych, pierwotnych warto?ci it_location: {df.it_location.nunique()}")
print(f"Liczba unikalnych, zakodowanych warto?ci it_location: {dfX.it_location.nunique()}")

## Encoding dla main_category

### 1. One Hot Encoder

Link do animacji pokazuj?cej zasad? dzia?ania one hot encodingu: https://brendanhasz.github.io/2019/03/04/target-encoding#one-hot-encoding

In [ ]:
dfX = df.copy()
dfX.head()

In [ ]:
enc = ce.OneHotEncoder(cols="main_category", use_cat_names=True)
dfX = enc.fit_transform(dfX)

dfX.head()

### 2. Leave One Out Encoding

Link do animacji pokazuj?cej zasad? dzia?ania leave one out encodingu: https://brendanhasz.github.io/2019/03/04/target-encoding#leave-one-out-target-encoding

Dzia?a podobnie do target encodingu, jednak przy wyliczniu ?redniej omija warto?? targetu dla liczonego wiersza - dzi?ki temu redukuje wp?yw outlier?w.

In [ ]:
dfX = df.copy()
dfX.head()

In [ ]:
encoder = ce.LeaveOneOutEncoder()
dfX['main_category'] = encoder.fit_transform(dfX['main_category'], dfX['price'])

dfX.head()

### 3. James-Stein Encoder

Podobny do target encodingu, tylko ?e tutaj stosowany jest wz?r na ?redni? wa?on?:

$JS_{i} = (1-B)*mean(y_i) + B*mean(y)$

gdzie $mean(y_i)$ oznacza ?redni? warto?? targetu, kiedy zmienna kodowana ma warto?? $i$, natomiast $B$ to wsp??czynnik reguluj?cy, aby kodowanie by?o mnie zale?ne od rozk?adu tageta

In [ ]:
dfX = df.copy()

js = ce.JamesSteinEncoder(dfX, cols = 'main_category')
dfX = js.fit_transform(dfX, dfX['price'])
dfX.head()

W tym przypadku kodowanie jest jednoznaczne:

In [ ]:
print(f"Liczba unikalnych warto?ci main_category po zakodowaniu: {dfX.main_category.nunique()}")
print(f"Liczba unikalnych warto?ci main_category w oryginale: {df.main_category.nunique()}")

## Uzupe?nianie brak?w

In [ ]:
dfX = df[['price', 'it_seller_rating', 'it_quantity']].copy()
dfX.head()

In [ ]:
dfX = dfX.head(50000).copy()
dfX.shape

### Jedna kolumna

In [ ]:
def insert_nans_and_impute_one_column(dfX, return_out_df):
    dfXX = dfX.copy()
    dfXX.loc[dfXX.sample(frac = 0.1).index, 'it_seller_rating'] = np.nan
    
    from sklearn.impute import KNNImputer
    
    imputer = KNNImputer(n_neighbors=5, weights= "distance")
    out = imputer.fit_transform(dfXX)

    from sklearn.metrics import mean_squared_error
    
    vec = dfXX.it_seller_rating.isna()

    rms = mean_squared_error(dfX['it_seller_rating'][vec], out[vec, 1], squared=False)
    
    if not return_out_df:
        return rms
    else:
        y_actual = dfX.loc[vec.to_numpy(), 'it_seller_rating'].to_numpy()
        y_predicted = out[vec, 1]
        
        return rms, y_actual, y_predicted 

In [ ]:
# pojedyncza pr?ba

insert_nans_and_impute_one_column(dfX, False)

In [ ]:
dfX

In [ ]:
# pojedyncza pr?ba wraz z wektorami

rms, y_actual, y_predicted = insert_nans_and_impute_one_column(dfX, True)

In [ ]:
y_actual_df = pd.DataFrame(y_actual)
y_actual_df.columns = ['y']

y_actual_df['type'] = 'actual'

In [ ]:
y_predicted_df = pd.DataFrame(y_predicted)
y_predicted_df.columns = ['y']

y_predicted_df['type'] = 'predicted'

In [ ]:
X = pd.concat([y_actual_df, y_predicted_df])
X.head()

Por?wnanie rozk?ad?w danych, kt?re zosta?y usuni?te oraz danych imputowanych

In [ ]:
sns.kdeplot(data=X, x="y", hue="type")
plt.title('it_seller_rating density - imputed and actual')

plt.show()

In [ ]:
sns.scatterplot(x = y_actual, y = y_predicted)
plt.xlabel('y_actual')
plt.ylabel('y_predicted')
plt.title('it_seller_rating')

plt.show()

In [ ]:
tab_one_column = []
for i in range(10):
    tab_one_column.append(insert_nans_and_impute_one_column(dfX, False))

tab_one_column = np.array(tab_one_column)

In [ ]:
print(f'?redna warto?? b??du ?redniokwadratowego wynosi: {tab_one_column.mean():.2f}, natomiast odchylenie standardowe: {tab_one_column.std():.2f}')

W przypadku imputowania jednej kolumny rozk?ad zmiennej zmieni? si? nieznacznie - jest lekko przesuni?ty w prawo oraz maksimum jest troch? wy?sze.

Na scatterplocie, kt?rego celem by?o por?wnanie prawdziwej oraz wyliczonej warto?ci zmiennych, wida? lini? prost?, jednak wi?kszo?? obserwacji znajduje si? bli?ej lewego dolnego rogu wykresu. Ponadto, imputer nie poradzi? sobie z du?ymi warto?ciami - np. dla prawdziwej warto?ci w okolicy 200000-300000, sporo obserwacji jest poni?ej 100000,

### Dwie kolumny

In [ ]:
def insert_nans_and_impute_two_columns(dfX, return_out_df):
    dfXX = dfX.copy()
    dfXX.loc[dfXX.sample(frac = 0.1).index, 'it_seller_rating'] = np.nan
    dfXX.loc[dfXX.sample(frac = 0.1).index, 'it_quantity'] = np.nan
    
    from sklearn.impute import KNNImputer
    
    imputer = KNNImputer(n_neighbors=5, weights= "distance")
    out = imputer.fit_transform(dfXX)
    
    from sklearn.metrics import mean_squared_error
    
    vec = dfXX.it_seller_rating.isna()

    rms = mean_squared_error(dfX['it_seller_rating'][vec], out[vec, 1], squared=False)
    
    if not return_out_df:
        return rms
    else:
        y_actual = dfX.loc[vec.to_numpy(), ['it_seller_rating', 'it_quantity']].to_numpy()
        y_predicted = out[vec, :]
        y_predicted = y_predicted[:, [1, 2]]
        
        return rms, y_actual, y_predicted

In [ ]:
# pojedyncza pr?ba

insert_nans_and_impute_two_columns(dfX, False)

In [ ]:
# pojedyncza pr?ba wraz z wynikami

rms, y_actual, y_predicted = insert_nans_and_impute_two_columns(dfX, True)

In [ ]:
y_actual_df = pd.DataFrame(y_actual)
y_actual_df.columns = ['y1', 'y2']

y_actual_df['type'] = 'actual'

In [ ]:
y_predicted_df = pd.DataFrame(y_predicted)
y_predicted_df.columns = ['y1', 'y2']

y_predicted_df['type'] = 'predicted'

In [ ]:
X = pd.concat([y_actual_df, y_predicted_df])
X.columns = ['it_seller_rating', 'it_quantity', 'type']
X.head()

Por?wnanie rozk?ad?w danych, kt?re zosta?y usuni?te oraz danych imputowanych

In [ ]:
sns.jointplot(data = X, x = 'it_seller_rating', y = 'it_quantity', hue = 'type', kind = 'kde')

plt.show()

In [ ]:
sns.scatterplot(x = X[X.type == 'actual'].it_seller_rating, y = X[X.type == 'predicted'].it_seller_rating)
plt.xlabel('actual')
plt.ylabel('predicted')
plt.title('it_seller_rating')

plt.show()

In [ ]:
sns.scatterplot(x = X[X.type == 'actual'].it_quantity, y = X[X.type == 'predicted'].it_quantity)
plt.xlabel('actual')
plt.ylabel('predicted')
plt.title('it_quantity')

plt.show()

In [ ]:
tab_two_columns = []
for i in range(10):
    tab_two_columns.append(insert_nans_and_impute_two_columns(dfX, False))

tab_two_columns = np.array(tab_two_columns)

In [ ]:
print(f'?redna warto?? b??du ?redniokwadratowego wynosi: {tab_two_columns.mean():.2f}, natomiast odchylenie standardowe: {tab_two_columns.std():.2f}')

W przypadku braku dw?ch kolumn, b??d ?redniokwadratowy okaza? si? jeszcze wi?kszy, natomiast odchylenie standardowe mniejsze.

Wynikowe rozk?ady wygl?daj? gorzej ni? w przypadku jednej kolumny. Obserwacje w przypadku kolumny *it_seller_rating* nadal s? roz?o?one bli?ej lewego dolnego rogu wykresu, jednak linia prosta (kt?ra wskazuje poprawne wyniki) jest mniej "wype?niona" ni? w przypadku poprzedniego eksperymentu. Ponadto, ?adna obserwacja nie zosta?a imputowana warto?ci? powy?ej 300000. Je?li chodzi o kolumn? *it_quantity* wida? wyra?n??lini? prost? i wydaje si?, ?e zosta?a ona do?? dobrze zimputowana.

### Podsumowanie

Narz?dzia do automatycznej imputacji s? wygodne w u?yciu, jednak w tym przypadku prowadz? do bardzo du?ych b??d?w. Przyjrzyjmy si?, jak r??ni?y si? b??dy w przypadku imputowania jednej oraz dw?ch kolumn.

In [ ]:
tab_one_column_df = pd.DataFrame(tab_one_column)
tab_one_column_df.columns = ['val']
tab_one_column_df['type'] = 'one column'

tab_two_columns_df = pd.DataFrame(tab_two_columns)
tab_two_columns_df.columns = ['val']
tab_two_columns_df['type'] = 'two columns'

X = pd.concat([tab_one_column_df, tab_two_columns_df])
X.columns = ['val', 'type']
X.head()

In [ ]:
sns.violinplot(data = X, y = 'val', x = 'type')
plt.title('RMSE')

plt.show()

B??d RMSE jest w wi?kszy w przypadku imputowania dw?ch kolumn. 

Ponadto, b??dy te maj? inne rozk?ady - b??dy przy imputacji jednej kolumny s? bardziej rozrzucone, podczas gdy w przypadku dw?ch kolumn warto?ci b??d?w roz?o?one s? bli?ej ?redniej (jednak ta ?rednia jest wy?sza ni? w przypadki pierwszego przypadku).